# S&P 500 Financial Analysis

End-to-end analysis of S&P 500 constituents including 
data cleaning, exploratory analysis, and SQL window functions.

**Dataset:** 503 companies · 14 original columns · 2017 snapshot  
**Tools:** Python · pandas · NumPy · matplotlib · seaborn · SQLite

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## 1. Loading & First Look

In [ ]:
url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies-financials/master/data/constituents-financials.csv"

df = pd.read_csv(url)
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.describe())
print(df.nlargest(5, 'Price/Earnings')[['Symbol','Name','Sector','Price/Earnings']])
print(df.nsmallest(5, 'Price/Earnings')[['Symbol','Name','Sector','Price/Earnings']])
print(df[df['Price/Earnings'] < 0][['Symbol','Name','Sector','Price/Earnings']])

## 2. Data Cleaning

Three issues identified during reconnaissance:
- 15 companies missing all financial metrics (systematic missingness)
- 102 missing Dividend Yield values — filled with 0 (companies that don't pay dividends)
- Price/Earnings outliers above 100 — flagged, not removed (economic outliers, not errors)
- SEC Filings column contains URLs with no analytical value — dropped

In [ ]:
df = (
    df
    .assign(
        data_quality = lambda x: np.where(x['Price'].isnull(), 'incomplete', 'ok'),
        pe_outlier   = lambda x: x['Price/Earnings'] > 100,
    )
    .fillna({'Dividend Yield': 0})
    .drop(columns=['SEC Filings'])
)
assert df.shape == (503, 15), f"Unexpected shape: {df.shape}"
assert df['Dividend Yield'].isnull().sum() == 0, "Dividend Yield still has nulls"
assert df['data_quality'].value_counts().get('incomplete', 0) == 15, "Expected 15 incomplete records"
assert df['pe_outlier'].value_counts().get(True, 0) == 18, "Expected 15 P/E outliers"
print(df.shape) 

## 3. Exploratory Analysis

**Finding 1:** S&P 500 representation is highly concentrated — 
a few sub-industries like Health Care Equipment dominate with 18 companies 
while most sub-industries have only one company large enough to qualify by market cap.

**Finding 2:** Health Care REITs trade at a median P/E of 77 vs Cable & Satellite 
at 4.3 — reflecting divergent investor expectations. The market prices in strong 
earnings growth for Real Estate while Cable & Satellite faces structural decline 
from streaming substitution.

In [ ]:
clean_df = df[df['data_quality'] == 'ok'].copy()
sector_counts = clean_df['Sector'].value_counts()

plt.figure(figsize=(10, 30))
sns.barplot(data=sector_counts.reset_index(), x='count', y='Sector')
plt.title('Number of Companies by Sector (Clean Data Only)')
plt.xlabel('Number of Companies')
plt.ylabel('Sector')
plt.tight_layout()
plt.show()

In [ ]:
median_pe = (
    clean_df[~clean_df['pe_outlier']]
    .groupby('Sector')['Price/Earnings']
    .median()
    .reset_index()
    .rename(columns={'Price/Earnings': 'median_PE'})
    .sort_values('median_PE', ascending=False)
)
print(median_pe)

## 4. SQL Analysis

*Coming next session — 10 window function queries on sp500.db*

In [ ]:
import sqlite3

conn = sqlite3.connect('sp500.db')
df.to_sql('sp500', conn, if_exists='replace', index=False)
print(f"Database created: {pd.read_sql_query('SELECT COUNT(*) as rows FROM sp500', conn).iloc[0,0]} rows loaded")

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('sp500.db')

def sql(query):
    return pd.read_sql_query(query, conn)

# test it
sql("SELECT Symbol, Name, Sector, Price FROM sp500 LIMIT 5")

## 4. SQL Analysis
Ten window function queries on the cleaned S&P 500 database.
All queries run on `sp500.db` using SQLite.

Key SQL concepts covered: ROW_NUMBER, RANK, DENSE_RANK, 
LAG, LEAD, NTILE, running totals, correlated subqueries, 
and CTE chaining.

### Query 1 — Basic Ranking by Price
Rank all S&P 500 companies by stock price descending using ROW_NUMBER().
Excludes companies with missing price data.

In [ ]:
sql("""
    SELECT 
        Symbol,
        Name, 
        Sector,
        Price,
        ROW_NUMBER() OVER (ORDER BY Price DESC) AS price_rank
    FROM sp500
    WHERE Price IS NOT NULL
""")

### Query 2 — RANK vs ROW_NUMBER
Compare RANK and ROW_NUMBER on Price/Earnings to illustrate 
how ties are handled differently. Excludes outliers and NULLs.

In [ ]:
sql("""
    SELECT
        Symbol,
        Name,
        [Price/Earnings],
        ROW_NUMBER() OVER (ORDER BY [Price/Earnings] DESC) AS rn,
        RANK() OVER (ORDER BY [Price/Earnings] DESC) AS rnk
    FROM sp500
    WHERE [Price/Earnings] IS NOT NULL AND pe_outlier = 0
    LIMIT 20
""")

### Query 3 — Top Company per Sector by Price
Uses PARTITION BY to rank companies within each sector independently.
Returns only the most expensive stock in each sector.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            Sector,
            Price,
            ROW_NUMBER() OVER (PARTITION BY Sector ORDER BY Price DESC) AS sector_price_rank
        FROM sp500
    )
    SELECT *
    FROM ranked
    WHERE sector_price_rank = 1
""")

### Query 4 — LAG and LEAD on Market Cap
Compares each company to the one ranked above and below it by market cap.
The $725B gap between Nvidia (rank 1) and Apple (rank 2) reflects 
the AI-driven surge in Nvidia's valuation.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            "Market Cap",
            ROW_NUMBER() OVER (ORDER BY "Market Cap" DESC) AS mkt_cap_rank,
            LAG(Name, 1) OVER (ORDER BY "Market Cap" DESC) AS prev_company,
            LEAD(Name, 1) OVER (ORDER BY "Market Cap" DESC) AS next_company,
            ROUND("Market Cap" - LAG("Market Cap", 1) OVER (ORDER BY "Market Cap" DESC), 2) AS mkt_cap_diff
        FROM sp500
        WHERE "Market Cap" IS NOT NULL
    )
    SELECT *
    FROM ranked
    LIMIT 15
""")

### Query 5 — Sector P/E Tiers with DENSE_RANK
Assigns growth tiers within each sector based on Price/Earnings.
Returns only tier 1 and 2 companies — the highest P/E names per sector.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            Sector,
            "Price/Earnings",
            DENSE_RANK() OVER (PARTITION BY Sector ORDER BY "Price/Earnings" DESC) AS tier
        FROM sp500
        WHERE "Price/Earnings" IS NOT NULL AND pe_outlier = 0
    )
    SELECT *
    FROM ranked
    WHERE tier <= 2
    ORDER BY Sector
""")

### Query 6 — Running Total of Market Cap
Cumulative market cap from largest to smallest company.
Finding: the top 20 companies account for 54% of total S&P 500 
market cap — the index is heavily concentrated at the top.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            "Market Cap",
            ROW_NUMBER() OVER (ORDER BY "Market Cap" DESC) AS mkt_cap_rank,
            SUM("Market Cap") OVER (ORDER BY "Market Cap" DESC) AS running_total,
            ROUND(SUM("Market Cap") OVER (ORDER BY "Market Cap" DESC) /
                  SUM("Market Cap") OVER () * 100, 2) AS pct_of_total
        FROM sp500
        WHERE "Market Cap" IS NOT NULL
    )
    SELECT *
    FROM ranked
    WHERE mkt_cap_rank <= 20
""")

### Query 7 — Running Average P/E
Tracks how the cumulative average P/E evolves as we move down 
the ranking. The declining average confirms that high P/E ratios 
are concentrated in a small number of companies.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            "Price/Earnings",
            ROW_NUMBER() OVER (ORDER BY "Price/Earnings" DESC) AS pe_rank,
            ROUND(AVG("Price/Earnings") OVER (ORDER BY "Price/Earnings" DESC), 2) AS running_avg_pe
        FROM sp500
        WHERE "Price/Earnings" IS NOT NULL AND pe_outlier = 0
    )
    SELECT *
    FROM ranked
    WHERE pe_rank <= 20
""")

### Query 8 — Price/Sales Quartiles vs P/E
Buckets companies into 4 quartiles by Price/Sales using NTILE(4).
Finding: companies in the top P/S quartile have nearly double 
the average P/E of the bottom quartile — the market assigns 
high growth expectations to both multiples simultaneously.

In [ ]:
sql("""
    WITH ranked AS (
        SELECT
            Symbol,
            Name,
            Sector,
            "Price/Earnings",
            NTILE(4) OVER (ORDER BY "Price/Sales" DESC) AS ps_quartile
        FROM sp500
        WHERE "Price/Sales" IS NOT NULL and pe_outlier = 0
    )
    SELECT ps_quartile,
           ROUND(AVG("Price/Earnings"), 2) AS avg_pe,
           COUNT(*) AS company_count
    FROM ranked
    GROUP BY ps_quartile
    ORDER BY ps_quartile
""")

### Query 9 — Companies Beating Their Sector Average Price
Correlated subquery returns companies whose price exceeds 
their sector average. Only 37% qualify — confirming that 
stock prices within sectors are right-skewed, with a few 
high-priced stocks pulling the mean above most companies.

In [ ]:
sql("""
    SELECT
        Symbol,
        Name,
        Sector,
        Price
    FROM sp500 a
    WHERE Price IS NOT NULL AND "Price"> (SELECT AVG(Price) from sp500 b WHERE b.Sector = a.Sector)
    ORDER BY Sector, Price DESC
""")

### Query 10 — Top 5 Sectors by Average P/E (CTE Chaining)
Chains two CTEs — sector_stats aggregates per-sector metrics, 
sector_ranked assigns P/E rankings. Returns the 5 sectors where 
the market expects the highest future earnings growth.

In [ ]:
sql("""
    WITH sector_stats AS (
        SELECT
            Symbol,
            Name,
            Sector,
            ROUND(AVG("Price/Earnings"), 2) AS avg_pe,
            ROUND(AVG(Price), 2) AS avg_price,
            COUNT(*) AS company_count
        FROM sp500
        WHERE "Price/Earnings" IS NOT NULL AND pe_outlier = 0 AND Price IS NOT NULL
        GROUP BY Sector
    ),
    sector_ranked AS (
        SELECT
            Sector,
            company_count,
            avg_price,
            avg_pe,
            ROW_NUMBER() OVER (ORDER BY avg_pe DESC) AS pe_rank
        FROM sector_stats
    )
    SELECT *
    FROM sector_ranked
    WHERE pe_rank <= 5
    ORDER BY pe_rank
""")